# Task 2: Exploratory Data Analysis — Tech Job Market
**CodeAlpha Data Analytics Internship**

**Questions we're asking of this dataset:**
1. Which skills are most in-demand overall, and how does that differ by role?
2. Which cities have the most job postings for each role?
3. What experience levels are most commonly requested?
4. Are there data quality issues (missing values, inconsistent formatting) to flag?


In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("india_job_market_2024_2026.csv")
df.shape


(5000, 17)

In [2]:
print(df.columns.tolist())

['Job_ID', 'Job_Title', 'Company', 'Company_Type', 'Industry', 'City', 'Location_Tier', 'Experience_Level', 'Job_Type', 'Work_Mode', 'Salary_LPA', 'Skills_Required', 'Education_Required', 'Openings', 'Applicants', 'Company_Rating', 'Date_Posted']


## 1. Structure and data types

In [3]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 17 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Job_ID              5000 non-null   str    
 1   Job_Title           5000 non-null   str    
 2   Company             5000 non-null   str    
 3   Company_Type        5000 non-null   str    
 4   Industry            5000 non-null   str    
 5   City                5000 non-null   str    
 6   Location_Tier       5000 non-null   str    
 7   Experience_Level    5000 non-null   str    
 8   Job_Type            5000 non-null   str    
 9   Work_Mode           5000 non-null   str    
 10  Salary_LPA          5000 non-null   float64
 11  Skills_Required     5000 non-null   str    
 12  Education_Required  5000 non-null   str    
 13  Openings            5000 non-null   int64  
 14  Applicants          5000 non-null   int64  
 15  Company_Rating      5000 non-null   float64
 16  Date_Posted      

In [4]:
df.head()


,Job_ID,Job_Title,Company,Company_Type,Industry,City,Location_Tier,Experience_Level,Job_Type,Work_Mode,Salary_LPA,Skills_Required,Education_Required,Openings,Applicants,Company_Rating,Date_Posted
0,IND2025000,Android Developer,Tech Mahindra,MNC,Information Technology,Remote,Remote,Senior (6-10 yrs),Full-Time,Remote,30.9,"Kotlin, Java, REST APIs",M.Tech/M.E.,3,276,4.0,2025-10-31
1,IND2025001,QA Engineer,Dream11,Indian Unicorn,Information Technology,Lucknow,Tier 2,Senior (6-10 yrs),Full-Time,Hybrid,58.6,"Selenium, Manual Testing, Postman, API Testing...",B.Tech/B.E.,3,325,4.0,2025-05-19
2,IND2025002,Business Analyst,HAL,PSU/Govt,EdTech,Remote,Remote,Senior (6-10 yrs),Full-Time,Remote,18.4,"JIRA, Excel, Power BI",MCA,5,559,3.6,2024-08-21
3,IND2025003,Cybersecurity Analyst,Groww,Startup,Information Technology,Mumbai,Tier 1,Mid (3-6 yrs),Full-Time,Hybrid,21.7,"Penetration Testing, Python, Ethical Hacking, ...",BCA,3,184,3.5,2026-03-18
4,IND2025004,Python Developer,Oracle,MNC,EdTech,Remote,Remote,Junior (1-3 yrs),Full-Time,Remote,8.0,"Docker, REST APIs, AWS, PostgreSQL",MCA,1,64,3.9,2024-10-25


## 2. Missing values check
Web-scraped data is almost always messier than a clean CSV dataset — check this honestly.

In [5]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})


,missing_count,missing_pct
Job_ID,0,0.0
Job_Title,0,0.0
Company,0,0.0
Company_Type,0,0.0
Industry,0,0.0
City,0,0.0
Location_Tier,0,0.0
Experience_Level,0,0.0
Job_Type,0,0.0
Work_Mode,0,0.0


## 3. Postings per role

In [6]:
df["Job_Title"].value_counts().head(10)

Job_Title
Software Engineer       336
Backend Developer       278
Full Stack Developer    260
Data Analyst            254
Java Developer          249
Python Developer        246
React Developer         224
Data Scientist          218
Business Analyst        198
Frontend Developer      193
Name: count, dtype: int64

## 4. Clean and split the skills column
Skills usually arrive as a comma-separated string — split into a proper list, then explode into one row per skill for counting.

In [7]:
df["skills_clean"] = df["Skills_Required"].fillna("").apply(
    lambda x: [s.strip().lower() for s in x.split(",")
 if s.strip()]
)
skills_exploded = df.explode("skills_clean")
skills_exploded = skills_exploded[skills_exploded["skills_clean"] != ""]

top_skills_overall = skills_exploded["skills_clean"].value_counts().head(20)
top_skills_overall

skills_clean
python           1579
rest apis         998
aws               867
sql               847
docker            836
java              636
postgresql        455
react             454
typescript        440
agile             429
javascript        419
mongodb           416
node.js           415
power bi          388
excel             387
kubernetes        385
system design     359
jira              346
statistics        343
tensorflow        325
Name: count, dtype: int64

## 5. Top skills by role
This is the comparison insight — how do required skills differ across the 3 roles?

In [8]:
for role in df["Job_Title"].unique():
    print(f"\n--- Top 10 skills for {role} ---")
    role_skills = skills_exploded[skills_exploded["Job_Title"] == role]
    print(role_skills["skills_clean"].value_counts().head(10))


--- Top 10 skills for Android Developer ---
skills_clean
rest apis          133
android sdk        131
kotlin             128
java               128
jetpack compose    122
firebase           121
Name: count, dtype: int64

--- Top 10 skills for QA Engineer ---
skills_clean
manual testing    114
api testing       108
postman           107
selenium          104
python            104
jira              102
Name: count, dtype: int64

--- Top 10 skills for Business Analyst ---
skills_clean
agile                     149
excel                     148
power bi                  147
sql                       145
requirements gathering    142
jira                      139
Name: count, dtype: int64

--- Top 10 skills for Cybersecurity Analyst ---
skills_clean
network security       82
ethical hacking        81
python                 79
penetration testing    78
siem                   73
Name: count, dtype: int64

--- Top 10 skills for Python Developer ---
skills_clean
rest apis     170
docker      

## 6. Location-wise demand

In [9]:
df["location_clean"] = df["City"].fillna("Not specified").str.strip()

df.groupby(["Job_Title", "location_clean"]).size() \
  .sort_values(ascending=False).head(20)

Job_Title                  location_clean
Software Engineer          Remote            143
Java Developer             Remote            106
Python Developer           Remote            105
Backend Developer          Remote            102
Data Analyst               Remote            101
Full Stack Developer       Remote             91
React Developer            Remote             86
Business Analyst           Remote             84
Data Scientist             Remote             83
DevOps Engineer            Remote             80
Android Developer          Remote             77
Node.js Developer          Remote             74
Data Engineer              Remote             72
Frontend Developer         Remote             71
Machine Learning Engineer  Remote             66
Product Manager            Remote             60
AI Engineer                Remote             57
QA Engineer                Remote             54
Technical Lead             Remote             54
Cloud Engineer             

## 7. Experience level patterns

In [10]:
df["Experience_Level"].value_counts().head(15)


Experience_Level
Mid (3-6 yrs)        1378
Junior (1-3 yrs)     1256
Fresher (0-1 yr)     1022
Senior (6-10 yrs)     831
Lead (10+ yrs)        513
Name: count, dtype: int64

## 8. Summary of findings

*(Fill this in after running the cells above — write 4-6 bullet points on what you actually found,
e.g. "Python and SQL appear in over 60% of Data Analyst postings but under 20% of Full Stack postings",
"Bangalore accounts for the largest share of postings across all 3 roles", etc.
These bullets are what you'll say out loud in your LinkedIn video.)*


In [11]:
df.to_csv("jobs_data_cleaned.csv", index=False)
print("Saved cleaned dataset for the visualization notebook.")


Saved cleaned dataset for the visualization notebook.
